The Jobicy dataset contains very long job descriptions. 
    
Some descriptions exceed 14,000 characters.

These descriptions contain commas and other special characters. 
    
Although the CSV is correctly formatted, MySQL Workbench's Import Wizard can encounter problems when processing such data.

So we have imported our Jobicy data using Jupyter Notebook instead of the MySQL Workbench Import Wizard.

Step 1: Install the MySQL Python connector

In [ ]:
%pip install mysql-connector-python

Step 2: Connect Jupyter Notebook to MySQL

In [1]:
import mysql.connector
from getpass import getpass

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter your MySQL password: "),
    database="jobavenir_db"
)

print("Connected to MySQL successfully!")

Enter your MySQL password:Sheetal@24 ········


Connected to MySQL successfully!


Step 3: Import the Jobicy CSV into MySQL

In [2]:
import pandas as pd
from pathlib import Path

# Your Jobicy CSV location
file_path = Path(
    r"C:\Users\belie\OneDrive\Desktop\ADT MySQL\Lab 7 Project Milestone\JobAvenir\data\cleaned\jobicy_jobs_clean.csv"
)

# Read Jobicy CSV
df = pd.read_csv(file_path, dtype=str)

# Display dataset information
print("Total Records:", len(df))
print("Columns:", df.columns.tolist())

display(df.head())

# Required columns
columns = [
    "job_id",
    "job_title",
    "company_name",
    "location",
    "description_clean",
    "publication_date"
]

# Ensure CSV contains the expected columns
assert list(df.columns) == columns, "CSV columns do not match!"

cursor = connection.cursor()

try:
    # Check whether data was already imported
    cursor.execute(
        "SELECT COUNT(*) FROM stg_jobicy_postings"
    )

    existing_rows = cursor.fetchone()[0]

    if existing_rows > 0:
        print(
            f"Import skipped: staging table already contains "
            f"{existing_rows} records."
        )

    else:
        # Convert missing values to None
        df = df.where(pd.notna(df), None)

        # SQL insert query
        insert_query = """
        INSERT INTO stg_jobicy_postings (
            job_id,
            job_title,
            company_name,
            location,
            description_clean,
            publication_date
        )
        VALUES (%s, %s, %s, %s, %s, %s)
        """

        # Convert DataFrame into tuples
        records = list(
            df[columns].itertuples(
                index=False,
                name=None
            )
        )

        # Insert records
        cursor.executemany(
            insert_query,
            records
        )

        connection.commit()

        print("Jobicy data imported successfully!")

    # Verify total records
    cursor.execute(
        "SELECT COUNT(*) FROM stg_jobicy_postings"
    )

    total = cursor.fetchone()[0]

    print("Total records in MySQL:", total)

except Exception as e:
    connection.rollback()
    print("Import failed:", e)
    raise

finally:
    cursor.close()

Total Records: 79
Columns: ['job_id', 'job_title', 'company_name', 'location', 'description_clean', 'publication_date']


,job_id,job_title,company_name,location,description_clean,publication_date
0,147852,"Principal Analyst, Data Integration",H1,USA,"At H1 , we believe access to the best healthca...",2026-10-31 03:30:19+00:00
1,158025,Senior Data Product Analyst,YipitData,USA,"Product · New York, NY (Remote-Friendly) About...",2026-10-30 16:26:51+00:00
2,157762,Senior Data Analyst (MedTech),YipitData,USA,About Us: YipitData is the leading market rese...,2026-10-29 05:48:09+00:00
3,157759,Senior Data Analyst,YipitData,USA,About Us: YipitData is the leading market rese...,2026-10-29 05:47:48+00:00
4,156941,Senior Data Analyst - Marketing,Supabase,Anywhere,About Supabase Supabase is the Postgres develo...,2026-10-25 10:45:07+00:00


Jobicy data imported successfully!
Total records in MySQL: 79
